# 한국어 BM25 검색과 Kiwi 형태소 분석

`rank-bm25`에 Kiwi 토크나이저를 직접 연결하고, Chroma 의미 검색과 weighted
RRF로 결합합니다. 보관된 `BM25Retriever`/`EnsembleRetriever`를 사용하지 않아
토큰화와 융합 점수가 어떻게 만들어지는지 그대로 확인할 수 있습니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain-core==1.6.3" "langchain-openai==1.6.2" \
#   "langchain-chroma==1.1.0" python-dotenv rank-bm25 kiwipiepy numpy


In [ ]:
import getpass
import os
import re
from collections import defaultdict
from uuid import uuid4

import numpy as np
from dotenv import load_dotenv
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from rank_bm25 import BM25Okapi

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

kiwi = Kiwi()
kiwi.tokenize("안녕하세요? 형태소 분석기 키위입니다")
CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


In [ ]:
texts = [
    "금융보험은 장기적인 자산 관리와 위험 대비를 목적으로 고안된 금융 상품입니다.",
    "금융저축보험은 규칙적인 저축으로 목돈을 마련하며 생명보험 기능도 제공합니다.",
    "저축금융보험은 저축과 금융을 통해 목돈 마련을 돕고 사망 보장도 제공합니다.",
    "금융저축산물보험은 장기 저축 목적과 축산물 제공 기능을 갖춘 특별 상품입니다.",
    "금융단폭격보험은 저축보다 위험 대비에 초점을 맞춘 상품입니다.",
    "금보험은 저축 성과를 극대화하며 노후 대비에 유리합니다.",
    "금융보씨 험한말은 검색 노이즈를 확인하기 위한 예문입니다.",
]
docs = [
    Document(page_content=text, metadata={"doc_id": f"doc-{index}"})
    for index, text in enumerate(texts)
]


def whitespace_tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


def kiwi_tokenize(text: str) -> list[str]:
    return [token.form for token in kiwi.tokenize(text)]


for doc in docs[:2]:
    print(kiwi_tokenize(doc.page_content))


## BM25 Runnable과 dense retriever


In [ ]:
def build_bm25_retriever(
    documents: list[Document],
    tokenizer,
    *,
    k: int = 4,
) -> RunnableLambda:
    index = BM25Okapi([tokenizer(doc.page_content) for doc in documents])

    def search(query: str) -> list[Document]:
        scores = index.get_scores(tokenizer(query))
        order = np.argsort(scores)[::-1][:k]
        return [
            Document(
                page_content=documents[position].page_content,
                metadata={
                    **documents[position].metadata,
                    "bm25_score": float(scores[position]),
                },
            )
            for position in order
        ]

    return RunnableLambda(search)


bm25 = build_bm25_retriever(docs, whitespace_tokenize)
kiwi_bm25 = build_bm25_retriever(docs, kiwi_tokenize)

dense_store = Chroma.from_documents(
    docs,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name=f"kiwi-dense-{uuid4().hex}",
    ids=[doc.metadata["doc_id"] for doc in docs],
    collection_configuration=CHROMA_CONFIGURATION,
)
dense = dense_store.as_retriever(search_kwargs={"k": 4})


## RRF 앙상블


In [ ]:
def make_rrf_retriever(
    left,
    right,
    *,
    weights: tuple[float, float],
    k: int = 4,
    c: int = 60,
) -> RunnableLambda:
    if not np.isclose(sum(weights), 1.0):
        raise ValueError("weights의 합은 1이어야 합니다.")

    def search(query: str) -> list[Document]:
        scores: defaultdict[str, float] = defaultdict(float)
        by_id: dict[str, Document] = {}
        for retriever, weight in zip((left, right), weights):
            for rank, doc in enumerate(retriever.invoke(query), start=1):
                doc_id = doc.metadata["doc_id"]
                by_id[doc_id] = doc
                scores[doc_id] += weight / (c + rank)
        ranked = sorted(scores, key=scores.get, reverse=True)[:k]
        return [
            Document(
                page_content=by_id[doc_id].page_content,
                metadata={**by_id[doc_id].metadata, "rrf_score": scores[doc_id]},
            )
            for doc_id in ranked
        ]

    return RunnableLambda(search)


bm25_dense_73 = make_rrf_retriever(bm25, dense, weights=(0.7, 0.3))
bm25_dense_37 = make_rrf_retriever(bm25, dense, weights=(0.3, 0.7))
kiwi_dense_73 = make_rrf_retriever(kiwi_bm25, dense, weights=(0.7, 0.3))
kiwi_dense_37 = make_rrf_retriever(kiwi_bm25, dense, weights=(0.3, 0.7))

retrievers = {
    "bm25": bm25,
    "kiwi_bm25": kiwi_bm25,
    "dense": dense,
    "bm25_dense_73": bm25_dense_73,
    "bm25_dense_37": bm25_dense_37,
    "kiwi_dense_73": kiwi_dense_73,
    "kiwi_dense_37": kiwi_dense_37,
}


In [ ]:
def print_search_results(query: str) -> None:
    print(f"\nQuery: {query}")
    for name, retriever in retrievers.items():
        result = retriever.invoke(query)
        print(f"{name:18}: {result[0].page_content if result else '결과 없음'}")


for query in [
    "금융보험",
    "금융 보험",
    "금융저축보험",
    "축산물 보험",
    "저축금융보험",
    "금융보씨 개인정보 조회",
]:
    print_search_results(query)


## 사용자 사전

복합명사를 하나의 고유명사로 등록한 뒤에는 **코퍼스도 다시 토큰화해 BM25
인덱스를 재구축**해야 합니다.


In [ ]:
kiwi.add_user_word("금융저축보험", "NNP", 0.0)
kiwi_bm25_with_dictionary = build_bm25_retriever(docs, kiwi_tokenize)
print(kiwi_tokenize("금융저축보험을 추천해 주세요"))
print(kiwi_bm25_with_dictionary.invoke("금융저축보험")[0].page_content)


## 선택 사항: KoNLPy 토크나이저 비교

KoNLPy는 Java/JVM 설정이 추가로 필요합니다. 핵심 실습과 분리하고 사용할 수 있는
환경에서만 실행합니다.


In [ ]:
# %pip install -qU konlpy

try:
    from konlpy.tag import Kkma, Okt

    kkma = Kkma()
    okt = Okt()
    sample = "안녕하세요? 형태소 분석기 테스트베드입니다."
    print("Kkma:", kkma.morphs(sample))
    print("Okt :", okt.morphs(sample))
    print("Kiwi:", kiwi_tokenize(sample))

    kkma_bm25 = build_bm25_retriever(docs, kkma.morphs)
    okt_bm25 = build_bm25_retriever(docs, okt.morphs)
    print("Kkma top-1:", kkma_bm25.invoke("금융저축보험")[0].page_content)
    print("Okt top-1 :", okt_bm25.invoke("금융저축보험")[0].page_content)
except Exception as exc:
    print("KoNLPy/JVM 환경이 없어 선택 실습을 건너뜁니다:", exc)
